# SOM Node Statistics

By: Ty Janoski

Standalone notebook for per-node precipitation and tropical cyclone analysis. Reads BMU assignments produced by `Z500_and_IVT_SOM_training.ipynb`.

In [40]:
import os
from itertools import combinations

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scienceplots  # noqa: F401
import xarray as xr
from matplotlib.cm import ScalarMappable
from matplotlib.colors import BoundaryNorm, ListedColormap
from metpy.plots import ctables
from scipy.stats import chi2_contingency, fisher_exact, mannwhitneyu

plt.style.use(["science", "nature", "grid"])
plt.rcParams["text.usetex"] = True

In [41]:
MOISTURE_VAR = "IVT"  # "IVT" or "tcwv"

xdim, ydim = 2, 2
n_nodes = xdim * ydim

_lbl = MOISTURE_VAR.lower()
fig_dir = f"figs/Z500-and-{_lbl}-SOM"
bmu_csv = f"data/som_2x2_bmus_{MOISTURE_VAR}.csv"
precip_path = "precip_data_and_tc_association_code/"
ibtracs_path = (
    "precip_data_and_tc_association_code/"
    "ibtracs.NA.list.v04r01.processed_6hrly.statslp3.csv"
)

os.makedirs(fig_dir, exist_ok=True)

## ASOS Precipitation by SOM Node

In [42]:
sites = {
    "JFK": np.load(f"{precip_path}jfk7_14_25.npy", allow_pickle=True),
    "LGA": np.load(f"{precip_path}lga7_14_25.npy", allow_pickle=True),
    "Central Park": np.load(f"{precip_path}cp7_14_25.npy", allow_pickle=True),
    "EWR": np.load(f"{precip_path}zewr_14_25.npy", allow_pickle=True),
}
precip_dfs = {}
for name, arr in sites.items():
    df = pd.DataFrame(arr, columns=["precip", "time"])
    df["time"] = pd.to_datetime(df["time"])
    df = df.set_index("time").sort_index()
    df["precip"] = pd.to_numeric(df["precip"], errors="coerce")
    precip_dfs[name] = df

bmu_df = pd.read_csv(bmu_csv)
bmu_df["timestamp"] = pd.to_datetime(bmu_df["timestamp"])
bmu_df["timestamp_local"] = (
    bmu_df["timestamp"]
    .dt.tz_localize("UTC")
    .dt.tz_convert("EST")
    .dt.tz_localize(None)
)

window_hours = 6
max_precip = []
for _, row in bmu_df.iterrows():
    t = row["timestamp_local"]
    lo = t - pd.Timedelta(hours=window_hours)
    hi = t + pd.Timedelta(hours=window_hours)
    site_maxes = [df.loc[lo:hi, "precip"].max() for df in precip_dfs.values()]
    valid = [v for v in site_maxes if not np.isnan(v)]
    max_precip.append(max(valid) if valid else 0.0)

bmu_df["max_precip_in"] = max_precip
print(
    f"Matched {len(bmu_df)} events | "
    f"range: {bmu_df['max_precip_in'].min():.2f}--"
    f"{bmu_df['max_precip_in'].max():.2f} in"
)

Matched 117 events | range: 0.00--3.62 in


In [43]:
node_stats = (
    bmu_df.groupby(["node_i", "node_j"])["max_precip_in"]
    .agg(count="count", mean="mean", median="median", std="std")
    .round(2)
)
print(node_stats)

               count  mean  median   std
node_i node_j                           
0      0          25  1.09    1.01  0.49
       1          33  0.85    0.77  0.58
1      0          35  1.06    1.03  0.49
       1          24  0.88    0.78  0.72


In [44]:
fig, axes = plt.subplots(ydim, xdim, figsize=(6, 4), constrained_layout=True, dpi=600)
bins = np.arange(0, 3.76, 0.25)

for i in range(xdim):
    for j in range(ydim):
        ax = axes[j, i]
        node_data = bmu_df[(bmu_df["node_i"] == i) & (bmu_df["node_j"] == j)][
            "max_precip_in"
        ].dropna()

        ax.hist(
            node_data, bins=bins, color="steelblue", alpha=0.9,
            edgecolor="white", linewidth=0.5,
        )
        n = len(node_data)
        if n > 0:
            median = node_data.median()
            ax.axvline(
                median, color="red", linestyle="--", linewidth=1,
                label=f'Median: {median:.2f}"',
            )
            ax.legend(fontsize=4, loc="upper right")

        ax.set_title(f"ASOS  ({i},{j})  N={n}", fontsize=6)
        ax.set_xlim(0, 3.75)
        ax.set_ylim(0, 12)
        ax.set_yticks(np.arange(0, 13, 2))
        ax.tick_params(axis="both", labelsize=5)
        ax.grid(True, linewidth=0.3, alpha=0.5, axis="y")
        if j == ydim - 1:
            ax.set_xlabel("Max Hourly Precip (in)", fontsize=5)
        if i == 0:
            ax.set_ylabel("Count", fontsize=5)

plt.suptitle(
    r"Max Hourly ASOS Precip ($\pm$6 hr window) by SOM Node",
    fontsize=8, y=1.02,
)
plt.savefig(
    f"{fig_dir}/Z500_and_{_lbl}_som_max_precip_histograms.png",
    bbox_inches="tight",
)
plt.close()
print(f"Saved to {fig_dir}/Z500_and_{_lbl}_som_max_precip_histograms.png")

Saved to figs/Z500-and-ivt-SOM/Z500_and_ivt_som_max_precip_histograms.png


## StageIV Gridded Precipitation by SOM Node

Per-node maximum hourly gridded precipitation from the NCEP Stage IV multi-sensor QPE
product (`/mnt/drive2/StageIV/stageiv_tristate_hourly.nc`). For each event the maximum
1-hour accumulation within a +-6 h window is taken over the NYC five-borough area
(40.5--40.9 N, 74.3--73.7 W). Precipitation is converted from mm to inches.

In [45]:
STAGEIV_NC = "/mnt/drive2/StageIV/stageiv_tristate_hourly.nc"

ds_s4 = xr.open_dataset(STAGEIV_NC, chunks={"time": 168})
s4_times = pd.DatetimeIndex(ds_s4["time"].values)
lat2d = ds_s4["latitude"].values
lon2d = ds_s4["longitude"].values

nyc_lat_min, nyc_lat_max = 40.5, 40.9
nyc_lon_min, nyc_lon_max = -74.3, -73.7
spatial_mask = (
    (lat2d >= nyc_lat_min)
    & (lat2d <= nyc_lat_max)
    & (lon2d >= nyc_lon_min)
    & (lon2d <= nyc_lon_max)
)

print(f"StageIV time range : {s4_times.min()} -- {s4_times.max()}")
print(
    f"NYC mask           : {int(spatial_mask.sum())} cells "
    f"({nyc_lat_min}--{nyc_lat_max} N, {nyc_lon_min}--{nyc_lon_max} W)"
)

StageIV time range : 2002-05-01 00:00:00 -- 2024-10-31 23:00:00
NYC mask           : 127 cells (40.5--40.9 N, -74.3---73.7 W)


In [46]:
MM_TO_IN = 1.0 / 25.4
_window = 6

all_needed: set[int] = set()
event_windows: list[list[int]] = []
for _, row in bmu_df.iterrows():
    t0 = row["timestamp"]
    mask = (s4_times >= t0 - pd.Timedelta(hours=_window)) & (
        s4_times <= t0 + pd.Timedelta(hours=_window)
    )
    idxs = np.where(mask)[0].tolist()
    event_windows.append(idxs)
    all_needed.update(idxs)

needed_idxs = sorted(all_needed)
print(f"Loading {len(needed_idxs)} unique StageIV time steps ...")

precip_raw = ds_s4["precipitation"].isel(time=needed_idxs).values
precip_raw = np.where(precip_raw < 0, np.nan, precip_raw)
precip_nyc = precip_raw[:, spatial_mask]
idx_to_sub = {orig: new for new, orig in enumerate(needed_idxs)}

max_precip_s4 = []
for idxs in event_windows:
    if idxs:
        sub = [idx_to_sub[i] for i in idxs]
        max_mm = np.nanmax(precip_nyc[sub, :])
        val = float(max_mm) * MM_TO_IN if not np.isnan(max_mm) else np.nan
        max_precip_s4.append(val)
    else:
        max_precip_s4.append(np.nan)

bmu_df["max_precip_s4_in"] = max_precip_s4
n_cov = bmu_df["max_precip_s4_in"].notna().sum()
print(f"Events with StageIV coverage : {n_cov}/{len(bmu_df)}")
print(
    f"StageIV precip range         : "
    f"{bmu_df['max_precip_s4_in'].min():.2f}--"
    f"{bmu_df['max_precip_s4_in'].max():.2f} in"
)

Loading 1287 unique StageIV time steps ...
Events with StageIV coverage : 99/117
StageIV precip range         : 0.40--3.86 in


In [47]:
s4_node_stats = (
    bmu_df.dropna(subset=["max_precip_s4_in"])
    .groupby(["node_i", "node_j"])["max_precip_s4_in"]
    .agg(count="count", mean="mean", median="median", std="std")
    .round(2)
)
print(s4_node_stats)

               count  mean  median   std
node_i node_j                           
0      0          22  1.53    1.34  0.58
       1          26  1.36    1.31  0.54
1      0          30  1.30    1.26  0.43
       1          21  1.45    1.34  0.71


In [48]:
fig, axes = plt.subplots(ydim, xdim, figsize=(6, 4), constrained_layout=True, dpi=600)
bins = np.arange(0, 4.01, 0.25)

for i in range(xdim):
    for j in range(ydim):
        ax = axes[j, i]
        node_data = bmu_df[(bmu_df["node_i"] == i) & (bmu_df["node_j"] == j)][
            "max_precip_s4_in"
        ].dropna()

        ax.hist(
            node_data, bins=bins, color="darkorange", alpha=0.9,
            edgecolor="white", linewidth=0.5,
        )
        n = len(node_data)
        if n > 0:
            median = node_data.median()
            ax.axvline(
                median, color="red", linestyle="--", linewidth=1,
                label=f'Median: {median:.2f}"',
            )
            ax.legend(fontsize=4, loc="upper right")

        ax.set_title(f"Stage IV  ({i},{j})  N={n}", fontsize=6)
        ax.set_xlim(0, 4.0)
        ax.set_ylim(0, 12)
        ax.set_yticks(np.arange(0, 13, 2))
        ax.tick_params(axis="both", labelsize=5)
        ax.grid(True, linewidth=0.3, alpha=0.5, axis="y")
        if j == ydim - 1:
            ax.set_xlabel("Max Hourly Precip (in)", fontsize=5)
        if i == 0:
            ax.set_ylabel("Count", fontsize=5)

plt.suptitle(
    r"Max Hourly Stage IV Precip ($\pm$6 hr window) by SOM Node",
    fontsize=8, y=1.02,
)
plt.savefig(
    f"{fig_dir}/Z500_and_{_lbl}_som_max_precip_histograms_stageiv.png",
    bbox_inches="tight",
)
plt.close()
print(f"Saved to {fig_dir}/Z500_and_{_lbl}_som_max_precip_histograms_stageiv.png")

Saved to figs/Z500-and-ivt-SOM/Z500_and_ivt_som_max_precip_histograms_stageiv.png


## ERA5 Total Precipitation by SOM Node

Per-node maximum hourly precipitation from the ERA5 reanalysis total precipitation field
(`/mnt/drive2/ERA5/NC_files/tp`). Six grid cells cover the five-borough core: two
latitude rows (40.50, 40.75 N) x three longitude columns (74.25, 74.00, 73.75 W).
ERA5 `tp` is in meters; values are converted to inches. Same +-6 hr window applied.

In [49]:
M_TO_IN = 1000.0 / 25.4

ERA5_TP_DIR = "/mnt/drive2/ERA5/NC_files/tp"
_era5_window = 6

era5_lat_max, era5_lat_min = 40.75, 40.5
era5_lon_min, era5_lon_max = -74.25, -73.75

max_precip_era5 = pd.Series(np.nan, index=bmu_df.index)

for year, grp in bmu_df.groupby(bmu_df["timestamp"].dt.year):
    fpath = f"{ERA5_TP_DIR}/era5_tp_NWHem_{year}.nc"
    if not os.path.exists(fpath):
        print(f"  {year}: file not found, skipping")
        continue

    ds_e5 = xr.open_dataset(fpath, engine="netcdf4")
    era5_times = pd.DatetimeIndex(ds_e5["valid_time"].values)
    tp_nyc = (
        ds_e5["tp"]
        .sel(
            latitude=slice(era5_lat_max, era5_lat_min),
            longitude=slice(era5_lon_min, era5_lon_max),
        )
        .load()
        .values
    )
    ds_e5.close()
    print(f"  {year}: {tp_nyc.shape} loaded, {len(grp)} event(s)")

    for idx, row in grp.iterrows():
        t0 = row["timestamp"]
        tmask = (era5_times >= t0 - pd.Timedelta(hours=_era5_window)) & (
            era5_times <= t0 + pd.Timedelta(hours=_era5_window)
        )
        t_idxs = np.where(tmask)[0]
        if len(t_idxs) == 0:
            continue
        max_m = float(np.nanmax(tp_nyc[t_idxs]))
        if not np.isnan(max_m):
            max_precip_era5.loc[idx] = max_m * M_TO_IN

bmu_df["max_precip_era5_in"] = max_precip_era5
n_cov = bmu_df["max_precip_era5_in"].notna().sum()
print(f"\nEvents with ERA5 coverage : {n_cov}/{len(bmu_df)}")
print(
    f"ERA5 precip range         : "
    f"{bmu_df['max_precip_era5_in'].min():.3f}--"
    f"{bmu_df['max_precip_era5_in'].max():.3f} in"
)

  1996: (4416, 2, 3) loaded, 6 event(s)
  1997: (4416, 2, 3) loaded, 1 event(s)
  1998: (4416, 2, 3) loaded, 2 event(s)
  1999: (4416, 2, 3) loaded, 2 event(s)
  2000: (4416, 2, 3) loaded, 5 event(s)
  2001: (4416, 2, 3) loaded, 2 event(s)
  2002: (4416, 2, 3) loaded, 3 event(s)
  2003: (4416, 2, 3) loaded, 4 event(s)
  2004: (4416, 2, 3) loaded, 7 event(s)
  2005: (4416, 2, 3) loaded, 2 event(s)
  2006: (4416, 2, 3) loaded, 8 event(s)
  2007: (4416, 2, 3) loaded, 6 event(s)
  2008: (4416, 2, 3) loaded, 7 event(s)
  2009: (4416, 2, 3) loaded, 2 event(s)
  2010: (4416, 2, 3) loaded, 3 event(s)
  2011: (4416, 2, 3) loaded, 4 event(s)
  2012: (4416, 2, 3) loaded, 6 event(s)
  2013: (4416, 2, 3) loaded, 5 event(s)
  2014: (4416, 2, 3) loaded, 8 event(s)
  2015: (4416, 2, 3) loaded, 3 event(s)
  2016: (4416, 2, 3) loaded, 1 event(s)
  2017: (4416, 2, 3) loaded, 4 event(s)
  2018: (4416, 2, 3) loaded, 8 event(s)
  2019: (4416, 2, 3) loaded, 5 event(s)
  2020: (4416, 2, 3) loaded, 1 event(s)


In [50]:
era5_node_stats = (
    bmu_df.dropna(subset=["max_precip_era5_in"])
    .groupby(["node_i", "node_j"])["max_precip_era5_in"]
    .agg(count="count", mean="mean", median="median", std="std")
    .round(3)
)
print(era5_node_stats)

               count   mean  median    std
node_i node_j                             
0      0          25  0.196   0.149  0.133
       1          33  0.171   0.122  0.138
1      0          35  0.253   0.223  0.170
       1          24  0.154   0.138  0.096


In [51]:
fig, axes = plt.subplots(ydim, xdim, figsize=(6, 4), constrained_layout=True, dpi=600)
era5_max = bmu_df["max_precip_era5_in"].max()
bins = np.arange(0, max(era5_max + 0.25, 2.01), 0.1)

for i in range(xdim):
    for j in range(ydim):
        ax = axes[j, i]
        node_data = bmu_df[(bmu_df["node_i"] == i) & (bmu_df["node_j"] == j)][
            "max_precip_era5_in"
        ].dropna()

        ax.hist(
            node_data, bins=bins, color="mediumpurple", alpha=0.9,
            edgecolor="white", linewidth=0.5,
        )
        n = len(node_data)
        if n > 0:
            median = node_data.median()
            ax.axvline(
                median, color="red", linestyle="--", linewidth=1,
                label=f'Median: {median:.2f}"',
            )
            ax.legend(fontsize=4, loc="upper right")

        ax.set_title(f"ERA5  ({i},{j})  N={n}", fontsize=6)
        ax.set_xlim(0, max(era5_max + 0.1, 2.0))
        ax.set_ylim(0, 12)
        ax.set_yticks(np.arange(0, 13, 2))
        ax.tick_params(axis="both", labelsize=5)
        ax.grid(True, linewidth=0.3, alpha=0.5, axis="y")
        if j == ydim - 1:
            ax.set_xlabel("Max Hourly Precip (in)", fontsize=5)
        if i == 0:
            ax.set_ylabel("Count", fontsize=5)

plt.suptitle(
    r"Max Hourly ERA5 Precip ($\pm$6 hr window) by SOM Node",
    fontsize=8, y=1.02,
)
plt.savefig(
    f"{fig_dir}/Z500_and_{_lbl}_som_max_precip_histograms_era5.png",
    bbox_inches="tight",
)
plt.close()
print(f"Saved to {fig_dir}/Z500_and_{_lbl}_som_max_precip_histograms_era5.png")

Saved to figs/Z500-and-ivt-SOM/Z500_and_ivt_som_max_precip_histograms_era5.png


## Tropical Cyclone Association by SOM Node

Flash flood event times are cross-referenced with IBTrACS to identify events where a
tropical system was present within the analysis domain (30--54 N, 100--60 W).

In [52]:
ibtracs = pd.read_csv(ibtracs_path)
ibtracs["ISO_TIME"] = pd.to_datetime(ibtracs["ISO_TIME"])

lat_min, lat_max = 30.0, 54.0
lon_min, lon_max = -100.0, -60.0

ibtracs_domain = ibtracs[
    (ibtracs["LAT"] >= lat_min)
    & (ibtracs["LAT"] <= lat_max)
    & (ibtracs["LON"] >= lon_min)
    & (ibtracs["LON"] <= lon_max)
].copy()

print(f"Total IBTrACS records : {len(ibtracs):,}")
print(f"Records in domain     : {len(ibtracs_domain):,}")
print(f"Unique storms         : {ibtracs_domain['SID'].nunique()}")

Total IBTrACS records : 64,320
Records in domain     : 12,035
Unique storms         : 1185


In [53]:
time_window_hours = 6
tc_associations = []

for _, row in bmu_df.iterrows():
    event_time = row["timestamp"]
    lo = event_time - pd.Timedelta(hours=time_window_hours)
    hi = event_time + pd.Timedelta(hours=time_window_hours)
    matching_tcs = ibtracs_domain[
        (ibtracs_domain["ISO_TIME"] >= lo) & (ibtracs_domain["ISO_TIME"] <= hi)
    ]

    if len(matching_tcs) > 0:
        storm_ids = matching_tcs["SID"].unique()
        tc_associations.append(
            {
                "timestamp": event_time,
                "node_i": row["node_i"],
                "node_j": row["node_j"],
                "tc_present": True,
                "n_storms": len(storm_ids),
                "storm_ids": ", ".join(storm_ids),
                "storm_status": (
                    matching_tcs["STAT"].mode().iloc[0]
                    if len(matching_tcs["STAT"].dropna()) > 0
                    else "Unknown"
                ),
            }
        )
    else:
        tc_associations.append(
            {
                "timestamp": event_time,
                "node_i": row["node_i"],
                "node_j": row["node_j"],
                "tc_present": False,
                "n_storms": 0,
                "storm_ids": "",
                "storm_status": "",
            }
        )

tc_df = pd.DataFrame(tc_associations)
n_tc = tc_df["tc_present"].sum()
print(f"TC-associated events: {n_tc}/{len(tc_df)} ({100 * n_tc / len(tc_df):.1f}%)")
for i in range(xdim):
    for j in range(ydim):
        nd = tc_df[(tc_df["node_i"] == i) & (tc_df["node_j"] == j)]
        tc_c = nd["tc_present"].sum()
        print(f"  Node ({i},{j}): {tc_c}/{len(nd)} ({100 * tc_c / len(nd):.1f}%)")

tc_events = tc_df[tc_df["tc_present"]].sort_values("timestamp")
print("\nTC-Associated Flash Flood Events:")
print("-" * 80)
for _, row in tc_events.iterrows():
    print(
        f"{row['timestamp'].strftime('%Y-%m-%d %H:%M')} | "
        f"Node ({row['node_i']},{row['node_j']}) | "
        f"Status: {row['storm_status']:>3} | {row['storm_ids']}"
    )

TC-associated events: 22/117 (18.8%)
  Node (0,0): 5/25 (20.0%)
  Node (0,1): 7/33 (21.2%)
  Node (1,0): 7/35 (20.0%)
  Node (1,1): 3/24 (12.5%)

TC-Associated Flash Flood Events:
--------------------------------------------------------------------------------
1996-07-13 13:00 | Node (1,0) | Status:  TS | 1996187N10326
1996-09-08 20:00 | Node (1,1) | Status:  EX | 1996237N14339
1997-07-16 00:00 | Node (0,1) | Status:  TS | 1997194N31286
1999-09-16 16:00 | Node (1,0) | Status:  HU | 1999251N15314
2001-06-17 15:00 | Node (0,0) | Status:  SS | 2001157N28265
2002-09-02 13:00 | Node (0,1) | Status:  TS | 2002245N29281
2004-09-08 10:00 | Node (0,0) | Status:  TD | 2004238N11325
2004-09-18 13:00 | Node (0,1) | Status:  EX | 2004247N10332
2004-09-28 21:00 | Node (0,1) | Status:  EX | 2004258N16300
2005-07-06 23:00 | Node (1,1) | Status:  TD | 2005185N18273
2005-10-14 21:00 | Node (0,1) | Status:  EX | 2005281N26303
2006-07-21 21:00 | Node (0,1) | Status:  EX | 2006200N32287
2007-06-04 13:00 | 

In [54]:
fig, axes = plt.subplots(1, 2, figsize=(8, 3.5), dpi=600, constrained_layout=True)

tc_counts = np.zeros((xdim, ydim))
non_tc_counts = np.zeros((xdim, ydim))
for i in range(xdim):
    for j in range(ydim):
        nd = tc_df[(tc_df["node_i"] == i) & (tc_df["node_j"] == j)]
        tc_counts[i, j] = nd["tc_present"].sum()
        non_tc_counts[i, j] = (~nd["tc_present"]).sum()

node_labels = [f"({i},{j})" for i in range(xdim) for j in range(ydim)]
x = np.arange(len(node_labels))
tc_flat = tc_counts.flatten()
non_tc_flat = non_tc_counts.flatten()
totals = tc_flat + non_tc_flat
tc_pct = 100 * tc_flat / totals

ax = axes[0]
ax.bar(x, non_tc_flat, 0.6, label="Non-TC", color="steelblue", alpha=0.9)
ax.bar(
    x, tc_flat, 0.6, bottom=non_tc_flat,
    label="TC-Associated", color="coral", alpha=0.9,
)
ax.set_xlabel("SOM Node", fontsize=7)
ax.set_ylabel("Number of Events", fontsize=7)
ax.set_title("Flash Flood Events by TC Association", fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels(node_labels, fontsize=6)
ax.legend(fontsize=6, loc="upper right")
ax.grid(True, linewidth=0.3, alpha=0.5, axis="y")

ax = axes[1]
bars = ax.bar(x, tc_pct, 0.6, color="coral", alpha=0.9, edgecolor="white")
for bar, pct, tc, total in zip(bars, tc_pct, tc_flat, totals):
    ax.text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
        f"{int(tc)}/{int(total)}", ha="center", va="bottom", fontsize=6,
    )
overall_pct = 100 * tc_flat.sum() / totals.sum()
ax.axhline(
    overall_pct, color="red", linestyle="--", linewidth=1,
    label=f"Overall: {overall_pct:.1f}\\%",
)
ax.set_xlabel("SOM Node", fontsize=7)
ax.set_ylabel(r"TC-Associated Events (\%)", fontsize=7)
ax.set_title("Percentage of Events with TC Influence", fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels(node_labels, fontsize=6)
ax.set_ylim(0, max(tc_pct) + 15)
ax.legend(fontsize=6, loc="upper right")
ax.grid(True, linewidth=0.3, alpha=0.5, axis="y")

plt.savefig(f"{fig_dir}/Z500_and_{_lbl}_som_tc_association.png", bbox_inches="tight")
plt.close()
print(f"Saved to {fig_dir}/Z500_and_{_lbl}_som_tc_association.png")

Saved to figs/Z500-and-ivt-SOM/Z500_and_ivt_som_tc_association.png


In [55]:
contingency_tc = np.array([tc_flat, non_tc_flat])

print("=" * 60)
print("CHI-SQUARE TEST: TC Association vs SOM Node")
print("=" * 60)
contingency_df = pd.DataFrame(
    {
        lbl: [int(tc), int(ntc)]
        for lbl, tc, ntc in zip(node_labels, tc_flat, non_tc_flat)
    },
    index=["TC", "Non-TC"],
)
contingency_df["Total"] = contingency_df.sum(axis=1)
print(contingency_df)

chi2, p_value, dof, expected = chi2_contingency(contingency_tc)
print(f"\nChi2={chi2:.3f}  dof={dof}  p={p_value:.4f}")
if expected.min() < 5:
    print("Warning: min expected count < 5")
if p_value < 0.05:
    print("-> REJECT H0: TC association differs across nodes.")
else:
    print("-> FAIL TO REJECT H0: no significant difference across nodes.")

print("\n" + "=" * 60)
print("PAIRWISE FISHER'S EXACT TESTS (Bonferroni-corrected)")
print("=" * 60)
pairs = list(combinations(range(n_nodes), 2))
alpha_corrected = 0.05 / len(pairs)
print(f"Bonferroni alpha: {alpha_corrected:.4f}\n")
for idx1, idx2 in pairs:
    table = np.array(
        [[tc_flat[idx1], tc_flat[idx2]], [non_tc_flat[idx1], non_tc_flat[idx2]]]
    )
    odds_ratio, p = fisher_exact(table)
    sig = "***" if p < alpha_corrected else ""
    print(
        f"{node_labels[idx1]} vs {node_labels[idx2]}: "
        f"OR={odds_ratio:.2f}, p={p:.4f} {sig}"
    )

CHI-SQUARE TEST: TC Association vs SOM Node
        (0,0)  (0,1)  (1,0)  (1,1)  Total
TC          5      7      7      3     22
Non-TC     20     26     28     21     95

Chi2=0.806  dof=3  p=0.8480
-> FAIL TO REJECT H0: no significant difference across nodes.

PAIRWISE FISHER'S EXACT TESTS (Bonferroni-corrected)
Bonferroni alpha: 0.0083

(0,0) vs (0,1): OR=0.93, p=1.0000 
(0,0) vs (1,0): OR=1.00, p=1.0000 
(0,0) vs (1,1): OR=1.75, p=0.7019 
(0,1) vs (1,0): OR=1.08, p=1.0000 
(0,1) vs (1,1): OR=1.88, p=0.4939 
(1,0) vs (1,1): OR=1.75, p=0.5059 


In [64]:
fig, ax = plt.subplots(
    figsize=(8, 5), subplot_kw={"projection": ccrs.PlateCarree()}, dpi=600
)
node_colors = {
    (0, 0): "tab:blue", (0, 1): "tab:orange",
    (1, 0): "tab:green", (1, 1): "tab:red",
}

for _, row in tc_events.iterrows():
    event_time = row["timestamp"]
    node_i, node_j = int(row["node_i"]), int(row["node_j"])
    color = node_colors[(node_i, node_j)]

    for sid in row["storm_ids"].split(", "):
        full_track = ibtracs[ibtracs["SID"] == sid].sort_values("ISO_TIME")
        if len(full_track) < 2:
            continue

        ax.plot(
            full_track["LON"], full_track["LAT"],
            color=color, linewidth=0.8, alpha=0.3, transform=ccrs.PlateCarree(),
        )

        wm = (
            full_track["ISO_TIME"] >= event_time - pd.Timedelta(hours=48)
        ) & (full_track["ISO_TIME"] <= event_time + pd.Timedelta(hours=48))
        wt = full_track[wm]
        if len(wt) > 1:
            ax.plot(
                wt["LON"], wt["LAT"],
                color=color, linewidth=1.2, alpha=0.85, transform=ccrs.PlateCarree(),
            )

        for delta_hrs in [-48, 48]:
            target = event_time + pd.Timedelta(hours=delta_hrs)
            diffs = (full_track["ISO_TIME"] - target).abs()
            idx = diffs.idxmin()
            if diffs[idx] <= pd.Timedelta(hours=6):
                pt = full_track.loc[idx]
                # ax.scatter(
                #     pt["LON"], pt["LAT"], color=color, s=60, marker="|",
                #     linewidths=1.5, zorder=6, transform=ccrs.PlateCarree(),
                # )

        ci = (full_track["ISO_TIME"] - event_time).abs().idxmin()
        cp = full_track.loc[ci]
        ax.scatter(
            cp["LON"], cp["LAT"], color=color, s=20, marker="o",
            edgecolor="black", linewidth=0.5, zorder=7, transform=ccrs.PlateCarree(),
        )

ax.plot(
    [lon_min, lon_max, lon_max, lon_min, lon_min],
    [lat_min, lat_min, lat_max, lat_max, lat_min],
    "k--", linewidth=1, transform=ccrs.PlateCarree(),
)
ax.scatter(
    -74.0, 40.7, color="black", s=100, marker="*",
    zorder=10, transform=ccrs.PlateCarree(),
)
ax.text(-73.5, 40.7, "NYC", fontsize=7, transform=ccrs.PlateCarree())
ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
ax.add_feature(cfeature.STATES.with_scale("50m"), linewidth=0.3)
ax.add_feature(cfeature.BORDERS, linewidth=0.3)
ax.set_extent([lon_min - 5, lon_max + 5, lat_min - 5, lat_max + 5])

legend_elements = [
    plt.Line2D(
        [0], [0], color=node_colors[(i, j)],
        linewidth=2, label=f"Node ({i},{j})",
    )
    for i in range(xdim)
    for j in range(ydim)
] + [
    plt.Line2D([0], [0], color="gray", linewidth=0.8, alpha=0.5, label="Full track"),
    plt.Line2D([0], [0], color="gray", linewidth=1.2, label=r"$\pm$48 hr window"),
]
ax.legend(handles=legend_elements, loc="lower right", fontsize=6)
ax.set_title(
    "TC Tracks During Flash Flood Events\n"
    r"(bold = $\pm$48 hr window, | = window bounds, $\bullet$ = event time)",
    fontsize=8,
)
plt.savefig(f"{fig_dir}/Z500_and_{_lbl}_som_tc_tracks.png", bbox_inches="tight")
plt.close()
print(f"Saved to {fig_dir}/Z500_and_{_lbl}_som_tc_tracks.png")

Saved to figs/Z500-and-ivt-SOM/Z500_and_ivt_som_tc_tracks.png


In [57]:
bmu_df_with_tc = bmu_df.merge(
    tc_df[["timestamp", "tc_present", "storm_ids"]], on="timestamp", how="left"
)

tc_precip = bmu_df_with_tc[bmu_df_with_tc["tc_present"]]["max_precip_in"].dropna()
non_tc_precip = (
    bmu_df_with_tc[~bmu_df_with_tc["tc_present"]]["max_precip_in"].dropna()
)

hdr = f"{'Category':<20} {'N':>6} {'Mean':>8} {'Median':>8} {'Std':>8}"
print(hdr)
print("-" * len(hdr))
for label, s in [("TC-Associated", tc_precip), ("Non-TC", non_tc_precip)]:
    print(
        f"{label:<20} {len(s):>6} {s.mean():>8.2f} {s.median():>8.2f} {s.std():>8.2f}"
    )

stat, p = mannwhitneyu(tc_precip, non_tc_precip, alternative="two-sided")
print(f"\nMann-Whitney U={stat:.1f}, p={p:.4f}")
if p < 0.05:
    print("-> Significant difference in precip intensity (TC vs non-TC).")
else:
    print("-> No significant difference in precip intensity.")

tc_df.to_csv("data/som_2x2_tc_associations.csv", index=False)
print("Saved TC association data to data/som_2x2_tc_associations.csv")

Category                  N     Mean   Median      Std
------------------------------------------------------
TC-Associated            22     1.17     1.02     0.77
Non-TC                   95     0.93     0.88     0.51

Mann-Whitney U=1219.5, p=0.2248
-> No significant difference in precip intensity.
Saved TC association data to data/som_2x2_tc_associations.csv


## Case Study: Hurricane Ida (1–2 September 2021)

Three-panel comparison of maximum 1-hour precipitation during Ida's remnants over the NYC/tri-state region. Panel (a) shows the four ASOS stations as colored dots; (b) shows the NCEP Stage IV 4-km QPE grid; (c) shows the ERA5 0.25° total precipitation grid with cell edges drawn explicitly to illustrate the coarse resolution. All panels share the same NWS-style color scale.

In [61]:
# ── MetPy precipitation colormap with fine NWS-style levels ──────────────────
nws_levels = [
    0.01, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50, 0.60,
    0.70, 0.80, 0.90, 1.00, 1.25, 1.50, 1.75, 2.00, 2.50, 3.00,
    3.50, 4.00, 5.00, 6.00, 8.00,
]
# Resample MetPy colormap to match the number of bins (levels - 1 = 24)
_n_bins = len(nws_levels) - 1
cmap_nws = ListedColormap(ctables.registry['precipitation']).resampled(_n_bins)
norm_nws = BoundaryNorm(nws_levels, ncolors=_n_bins, clip=False)

# ── Ida time window ───────────────────────────────────────────────────────────
ida_start = pd.Timestamp("2021-09-01 00:00:00")
ida_end = pd.Timestamp("2021-09-02 23:59:00")

# ── Map extent (shared by all three panels) ───────────────────────────────────
map_lon_min, map_lon_max = -74.8, -73.4
map_lat_min, map_lat_max = 40.3, 41.2

# ── Panel (a): ASOS — max hourly precip during Ida ───────────────────────────
station_coords = {
    "JFK": (40.6413, -73.7781),
    "LGA": (40.7769, -73.8740),
    "Central Park": (40.7829, -73.9654),
    "EWR": (40.6895, -74.1745),
}
# Text label, anchor alignment, and (dlon, dlat) offset for each station
station_labels = {
    "JFK": ("JFK", "left", 0.05, -0.05),
    "LGA": ("LGA", "left", 0.05, 0.02),
    "Central Park": ("CP", "right", -0.05, 0.03),
    "EWR": ("EWR", "left", 0.05, 0.02),
}

station_ida_max = {}
for name, df in precip_dfs.items():
    w = df.loc[ida_start:ida_end, "precip"]
    station_ida_max[name] = float(w.max()) if len(w) > 0 else np.nan

print("ASOS max hourly precip during Ida:")
for name, val in station_ida_max.items():
    print(f"  {name:<15}: {val:.2f} in")

# ── Panel (b): Stage IV — max hourly precip, masked to analysis domain ────────
# Analysis domain: nyc_lat_min/max, nyc_lon_min/max (127 cells via spatial_mask)
ida_s4_mask = (s4_times >= ida_start) & (s4_times <= ida_end)
ida_s4_raw = ds_s4["precipitation"].isel(time=np.where(ida_s4_mask)[0]).values
ida_s4_raw = np.where(ida_s4_raw < 0, np.nan, ida_s4_raw)
ida_s4_max_in = np.nanmax(ida_s4_raw, axis=0) / 25.4  # mm → in

# ── Panel (c): ERA5 — max hourly precip, exact analysis domain ───────────────
# Analysis domain: era5_lat_min/max, era5_lon_min/max (same variables as above)
ds_e5_ida = xr.open_dataset(f"{ERA5_TP_DIR}/era5_tp_NWHem_2021.nc", engine="netcdf4")
era5_times_2021 = pd.DatetimeIndex(ds_e5_ida["valid_time"].values)
ida_e5_idxs = np.where((era5_times_2021 >= ida_start) & (era5_times_2021 <= ida_end))[0]

e5_da = (
    ds_e5_ida["tp"]
    .sel(
        latitude=slice(era5_lat_max, era5_lat_min),
        longitude=slice(era5_lon_min, era5_lon_max),
    )
    .isel(valid_time=ida_e5_idxs)
    .load()
)
e5_lats = e5_da.latitude.values  # decreasing
e5_lons = e5_da.longitude.values  # increasing
ds_e5_ida.close()

ida_e5_max_in = np.nanmax(e5_da.values, axis=0) * M_TO_IN  # m → in
e5_lon2d, e5_lat2d = np.meshgrid(e5_lons, e5_lats)

print(
    f"\nERA5 patch: {ida_e5_max_in.shape}  "
    f"range {np.nanmin(ida_e5_max_in):.3f}--{np.nanmax(ida_e5_max_in):.3f} in"
)

# ── Build 3-panel figure ──────────────────────────────────────────────────────
fig, axes = plt.subplots(
    1,
    3,
    figsize=(10, 3.5),
    subplot_kw={"projection": ccrs.PlateCarree()},
    constrained_layout=True,
    dpi=300,
)
# Reserve top 12 % for the suptitle to eliminate excess whitespace
fig.get_layout_engine().set(rect=(0, 0, 1, 0.88))

for ax in axes:
    ax.set_extent([map_lon_min, map_lon_max, map_lat_min, map_lat_max])
    ax.add_feature(cfeature.LAND.with_scale("10m"), facecolor="#ebebeb", zorder=0)
    ax.add_feature(cfeature.OCEAN.with_scale("10m"), facecolor="#dde8f2", zorder=0)
    ax.add_feature(cfeature.COASTLINE.with_scale("10m"), linewidth=0.5, zorder=4)
    ax.add_feature(cfeature.STATES.with_scale("10m"), linewidth=0.4, zorder=4)
    # NYC reference star
    ax.scatter(
        -74.0,
        40.7,
        color="black",
        s=20,
        marker="*",
        zorder=5,
        transform=ccrs.PlateCarree(),
    )

for ax, title in zip(
    axes,
    [
        "(a) ASOS Stations",
        "(b) Stage IV (4-km)",
        r"(c) ERA5 ($0.25^{\circ}$)",
    ],
):
    ax.set_title(title, fontsize=7)

# ── (a) ASOS ──────────────────────────────────────────────────────────────────
ax = axes[0]
for name, (lat, lon) in station_coords.items():
    val = station_ida_max.get(name, np.nan)
    ax.scatter(
        [lon],
        [lat],
        c=[val],
        cmap=cmap_nws,
        norm=norm_nws,
        s=70,
        marker="o",
        edgecolors="k",
        linewidths=0.5,
        zorder=5,
        transform=ccrs.PlateCarree(),
    )
    label, ha, dlon, dlat = station_labels[name]
    ax.text(
        lon + dlon,
        lat + dlat,
        label,
        fontsize=5,
        ha=ha,
        transform=ccrs.PlateCarree(),
        zorder=6,
    )

# ── (b) Stage IV — only colour cells inside the analysis domain ───────────────
ax = axes[1]
plot_s4 = np.where(spatial_mask, ida_s4_max_in, np.nan)  # mask to analysis domain
plot_s4 = np.where(plot_s4 >= nws_levels[0], plot_s4, np.nan)  # hide trace amounts
ax.pcolormesh(
    lon2d,
    lat2d,
    plot_s4,
    cmap=cmap_nws,
    norm=norm_nws,
    shading="auto",
    zorder=2,
    transform=ccrs.PlateCarree(),
)

# ── (c) ERA5 — only the analysis-domain cells, edges drawn ────────────────────
ax = axes[2]
plot_e5 = np.where(ida_e5_max_in >= nws_levels[0], ida_e5_max_in, np.nan)
ax.pcolormesh(
    e5_lon2d,
    e5_lat2d,
    plot_e5,
    cmap=cmap_nws,
    norm=norm_nws,
    shading="auto",
    zorder=2,
    edgecolors="gray",
    linewidths=0.4,
    transform=ccrs.PlateCarree(),
)

# ── Shared colorbar ───────────────────────────────────────────────────────────
sm = ScalarMappable(cmap=cmap_nws, norm=norm_nws)
sm.set_array([])
cbar = fig.colorbar(
    sm, ax=axes.tolist(), orientation="vertical", pad=0.02, shrink=0.85, extend="max"
)
cbar.set_label("Max Hourly Precip (in)", fontsize=7)
cbar.ax.tick_params(labelsize=6)

fig.suptitle(
    "Hurricane Ida: Maximum Hourly Precipitation by Source\n1--2 September 2021",
    fontsize=8, y=0.9
)
plt.savefig(f"{fig_dir}/ida_precip_three_sources.png", bbox_inches="tight")
plt.close()
print(f"Saved to {fig_dir}/ida_precip_three_sources.png")

ASOS max hourly precip during Ida:
  JFK            : 0.80 in
  LGA            : 2.80 in
  Central Park   : 3.47 in
  EWR            : 3.62 in

ERA5 patch: (2, 3)  range 0.292--0.440 in


/tmp/ipykernel_427025/3970175481.py:49: RuntimeWarning: All-NaN slice encountered
  ida_s4_max_in = np.nanmax(ida_s4_raw, axis=0) / 25.4  # mm → in


Saved to figs/Z500-and-ivt-SOM/ida_precip_three_sources.png
